In [10]:
from selenium import webdriver
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

url = "https://www.nykaa.com/search/result/?q=beauty&root=search&searchType=Manual&sourcepage=home"

# Open Chrome
driver = webdriver.Chrome()

# Open Nykaa
driver.get(url)

# Wait for products to load
time.sleep(5)

# Get page source
soup = BeautifulSoup(driver.page_source, "html.parser")

# Find product titles
titles = soup.find_all("h2", class_="css-xrzmfa")

print("Number of products:", len(titles))

# Check first product's text
first_product = titles[0].find_parent("div")

print("\nFIRST PRODUCT TEXT:")
print(first_product.get_text(" ", strip=True))

# Empty list
products = []

for product in titles[:20]:

    # Product Name
  
    product_name = product.get_text(strip=True)

    # Get product card text
    parent = product.parent
    text = parent.get_text(" ", strip=True)

    # MRP
   
    mrp_match = re.search(r"Regular price ₹(\d+)", text)

    if mrp_match:
        mrp = mrp_match.group(1)
    else:
        mrp = "N/A"

   
    # Selling Price
    
    price_match = re.search(r"Discounted price ₹(\d+)", text)

    if price_match:
        price = price_match.group(1)
    else:
        price = "N/A"

  
    # Discount
  
    discount_match = re.search(r"(\d+)% Off", text)

    if discount_match:
        discount = discount_match.group(1) + "%"
    else:
        discount = "N/A"

    # Brand
    
    brand_match = re.match(r"(\S+)", product_name)

    if brand_match:
        brand = brand_match.group(1)
    else:
        brand = "N/A"

  
    # Rating
   
    rating = "N/A"

    current = product

    for i in range(5):

        if current:

            current_text = current.get_text(" ", strip=True)

            rating_match = re.search(r"\b[0-5]\.\d\b", current_text)

            if rating_match:
                rating = rating_match.group(0)
                break

            current = current.parent

    # Reviews

    reviews_match = re.search(r"\(\s*(\d+)\s*\)", text)

    if reviews_match:
        reviews = reviews_match.group(1)
    else:
        reviews = "N/A"

    # Product URL
  
    link = product.find_parent("a", href=True)

    if link:

        href = link["href"]

        if href.startswith("http"):
            product_url = href
        else:
            product_url = "https://www.nykaa.com" + href

    else:
        product_url = "N/A"

    # Category
  
    category = "Beauty"

    # Store data

    products.append({
        "Product Name": product_name,
        "Brand": brand,
        "MRP": mrp,
        "Selling Price": price,
        "Discount": discount,
        "Rating": rating,
        "Reviews": reviews,
        "Product URL": product_url,
        "Category": category
    })


# Close browser
driver.quit()

# Convert to DataFrame
df = pd.DataFrame(products)

# Display data
print("\nProduct Data:")
print(df)

# Save to CSV
df.to_csv("nykaa_products.csv", index=False)

print("\nCSV file saved successfully!")

Number of products: 20

FIRST PRODUCT TEXT:
TIRTIR Mask Fit Red Mini Cushion Foundation ₹1300 ₹1105 15% Off Regular price ₹1300. Discounted price ₹1105. 15% Off. ( 2408 ) 25 shades

Product Data:
                                         Product Name       Brand   MRP  \
0         TIRTIR Mask Fit Red Mini Cushion Foundation      TIRTIR  1300   
1   Swiss Beauty Bold Matt Lip Liner Pencil, Long-...       Swiss    69   
2   Insight Cosmetics Concealer, Buildabe Coverage...     Insight   135   
3   Feel Good Club Classic Makeup Blender Beauty S...        Feel   105   
4   Miduty Pigment Clear Glutathione Hyaluronic Ac...      Miduty   N/A   
5   Beauty of Joseon Relief Sun Aqua-Fresh Add SPF...      Beauty   N/A   
6   Dot & Key 10% Vitamin C + E Face Serum With 5%...         Dot   499   
7   Insight Cosmetics 24 Hrs Non Transfer Matte Li...     Insight   310   
8           Rare Beauty Soft Pinch Matte Liquid Blush        Rare   N/A   
9   Dot & Key Cica Calming Mattifying Sunscreen In... 

In [11]:
df = pd.DataFrame(products)

df.to_csv("nykaa_beauty_products.csv", index=False)

print("CSV file created successfully!")
print("Total products saved:", len(df))

CSV file created successfully!
Total products saved: 20


In [12]:
df

,Product Name,Brand,MRP,Selling Price,Discount,Rating,Reviews,Product URL,Category
0,TIRTIR Mask Fit Red Mini Cushion Foundation,TIRTIR,1300,1105,15%,N/A,2408,https://www.nykaa.com/tirtir-mask-fit-red-mini...,Beauty
1,"Swiss Beauty Bold Matt Lip Liner Pencil, Long-...",Swiss,69,66,5%,N/A,313242,https://www.nykaa.com/swiss-beauty-bold-matt-l...,Beauty
2,"Insight Cosmetics Concealer, Buildabe Coverage...",Insight,135,131,3%,N/A,38535,https://www.nykaa.com/insight-cosmetics-concea...,Beauty
3,Feel Good Club Classic Makeup Blender Beauty S...,Feel,105,60,43%,N/A,2355,https://www.nykaa.com/feel-good-club-classic-m...,Beauty
4,Miduty Pigment Clear Glutathione Hyaluronic Ac...,Miduty,N/A,N/A,N/A,N/A,245,https://www.nykaa.com/miduty-skin-shield-mitop...,Beauty
5,Beauty of Joseon Relief Sun Aqua-Fresh Add SPF...,Beauty,N/A,N/A,N/A,N/A,5431,https://www.nykaa.com/beauty-of-joseon-relief-...,Beauty
6,Dot & Key 10% Vitamin C + E Face Serum With 5%...,Dot,499,429,14%,N/A,63254,https://www.nykaa.com/dot-key-10percent-vitami...,Beauty
7,Insight Cosmetics 24 Hrs Non Transfer Matte Li...,Insight,310,295,5%,N/A,545160,https://www.nykaa.com/insight-cosmetics-24-hrs...,Beauty
8,Rare Beauty Soft Pinch Matte Liquid Blush,Rare,N/A,N/A,N/A,N/A,390,https://www.nykaa.com/rare-beauty-soft-pinch-m...,Beauty
9,Dot & Key Cica Calming Mattifying Sunscreen In...,Dot,445,387,13%,N/A,33451,https://www.nykaa.com/dot-key-cica-calming-nia...,Beauty
